# Covariance Generators and Validation

This notebook validates matrix symmetry, positive definiteness, Cholesky factorization, empirical covariance convergence, and easy-case parameter recovery before running the main experiments.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy.linalg import cholesky

from src.covariance_models import covariance_from_family, positions, validation_record
from src.fitting import fit_covariance_model
from src.metrics import relative_frobenius_error
from src.simulation import generate_realizations, sample_covariance


In [ ]:
n = 20
x = positions(n, 0.0, 10.0)
models = [
    ('diagonal', {'sigma': 1.0}),
    ('cs', {'sigma': 1.0, 'rho': 0.5}),
    ('rbf', {'sigma': 1.0, 'ell': 1.0, 'noise': 0.05}),
]

records = []
for family, params in models:
    Sigma = covariance_from_family(family, n, params, x=x)
    cholesky(Sigma, lower=True)
    records.append(validation_record(family, params, Sigma))

validation = pd.DataFrame(records)
validation.to_csv(ROOT / 'results/phase1/tables/model_validation.csv', index=False)
validation


In [ ]:
rng = np.random.default_rng(20260831)
Sigma_true = covariance_from_family('rbf', n, {'sigma': 1.0, 'ell': 1.5, 'noise': 0.05}, x=x)
Y = generate_realizations(Sigma_true, M=100_000, rng=rng)
Sigma_sample = sample_covariance(Y)
relative_frobenius_error(Sigma_sample, Sigma_true)


In [ ]:
rng = np.random.default_rng(20260832)
Y_easy = generate_realizations(Sigma_true, M=3000, rng=rng)
fit = fit_covariance_model(Y_easy, 'rbf', x=x, seed=20260833, n_restarts=10)
fit.parameters, relative_frobenius_error(fit.Sigma_hat, Sigma_true), fit.optimizer_success, fit.message
